# POC: Multi-Agent Collaboration via Apache Fluss & Gemini (v3)

This version uses a **Simplified Split-Cell** design to ensure visibility and responsiveness in all Jupyter environments.

## 1. Setup & Environment

In [ ]:
import fluss
import os
from datetime import datetime

# Get the exact path of the loaded compiled _fluss extension
extension_path = fluss._fluss.__file__
mod_time = os.path.getmtime(extension_path)

print(f"Loaded Extension: {extension_path}")
print(f"Compiled At: {datetime.fromtimestamp(mod_time)}")

In [ ]:
import os
import asyncio
import pyarrow as pa
import pandas as pd
from datetime import datetime
import time
import requests
import json
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    with open("../containerclaw/secrets/gemini_api_key.txt", "r") as f:
        GEMINI_API_KEY = f.read().strip()

print("✅ Environment Ready.")

In [ ]:
async def setup_chatroom():
    config = fluss.Config({"bootstrap.servers": "127.0.0.1:9123"})
    conn = await fluss.FlussConnection.create(config)
    admin = await conn.get_admin()
    
    db_name = "containerclaw"
    table_name = "chatroom"
    table_path = fluss.TablePath(db_name, table_name)
    
    await admin.create_database(db_name, ignore_if_exists=True)
    await admin.drop_table(table_path, ignore_if_not_exists=True)
    
    schema = fluss.Schema(
        pa.schema([
            pa.field("ts", pa.int64()), 
            pa.field("actor_id", pa.string()), 
            pa.field("content", pa.string())
        ])
    )
    table_descriptor = fluss.TableDescriptor(schema, bucket_count=1)
    
    await admin.create_table(table_path, table_descriptor)
    print(f"Created shared chatroom table: {table_path}")
    return conn, table_path

conn, table_path = await setup_chatroom()

## 2. Gemini Agent Loop

Run this cell to start the autonomous agents. They will listen to the Fluss stream and chime in when necessary.

In [ ]:
class GeminiAgent:
    def __init__(self, agent_id, persona):
        self.agent_id = agent_id
        self.persona = persona
        self.history = set()
        self.model = "gemini-3-flash-preview"

    async def _call_gemini(self, transcript):
        url = f"https://generativelanguage.googleapis.com/v1beta/models/{self.model}:generateContent?key={GEMINI_API_KEY}"
        system_instruction = (
            f"You are {self.agent_id}. Role: {self.persona}. "
            "You are in a shared chatroom. Watch the transcript. "
            "If the Human or another Agent says something interesting, respond. "
            "If you have nothing to add, or if you just spoke, respond with [WAIT]."
        )
        payload = {
            "contents": [{"role": "user", "parts": [{"text": f"SYSTEM: {system_instruction}\n\nTRANSCRIPT:\n{transcript}"}]}]
        }
        try:
            res = requests.post(url, json=payload, timeout=10)
            return res.json()['candidates'][0]['content']['parts'][0]['text'].strip() if res.status_code == 200 else None
        except: return None

    async def run(self, table):
        writer = table.new_append().create_writer()
        scanner = await table.new_scan().create_record_batch_log_scanner()
        scanner.subscribe(bucket_id=0, start_offset=0)
        
        print(f"🔹 Agent [{self.agent_id}] Online.")
        
        while True:
            poll = scanner.poll_arrow(timeout_ms=500)
            if poll.num_rows > 0:
                df = poll.to_pandas()
                new_msgs = []
                for _, row in df.iterrows():
                    key = f"{row['ts']}-{row['actor_id']}"
                    if key not in self.history:
                        self.history.add(key)
                        new_msgs.append(f"{row['actor_id']}: {row['content']}")
                        print(f"[{row['actor_id']}]: {row['content']}")

                # Only respond if the LAST message wasn't from us
                if new_msgs and df.iloc[-1]['actor_id'] != self.agent_id:
                    await asyncio.sleep(1.5) # Breath
                    resp = await self._call_gemini("\n".join(new_msgs))
                    if resp and "[WAIT]" not in resp:
                        print(f"🤖 [{self.agent_id}]: {resp}")
                        batch = pa.RecordBatch.from_arrays([
                            pa.array([int(time.time() * 1000)], type=pa.int64()),
                            pa.array([self.agent_id], type=pa.string()),
                            pa.array([resp], type=pa.string())
                        ], schema=pa.schema([pa.field("ts", pa.int64()), pa.field("actor_id", pa.string()), pa.field("content", pa.string())]))
                        writer.write_arrow_batch(batch)
                        await writer.flush()
            await asyncio.sleep(1)

In [ ]:
# START AGENTS HERE
table = await conn.get_table(table_path)
coder = GeminiAgent("Agent-1-Coder", "Expert Python Dev.")
reviewer = GeminiAgent("Agent-2-Reviewer", "Security Specialist.")

print("Starting autonomous agents. Use the cell below to send messages.")
await asyncio.gather(coder.run(table), reviewer.run(table))

## 3. Human Participant (Chat Here)

Run this cell repeatedly to send messages into the Fluss stream. Each time you run it, it will prompt you for input.

In [ ]:
async def send_message(content):
    table = await conn.get_table(table_path)
    writer = table.new_append().create_writer()
    
    batch = pa.RecordBatch.from_arrays([
        pa.array([int(time.time() * 1000)], type=pa.int64()),
        pa.array(["Human"], type=pa.string()),
        pa.array([content], type=pa.string())
    ], schema=pa.schema([pa.field("ts", pa.int64()), pa.field("actor_id", pa.string()), pa.field("content", pa.string())]))
    
    writer.write_arrow_batch(batch)
    await writer.flush()
    print(f"✅ Sent: {content}")

# Type and execute!
msg = input("Human Message: ")
await send_message(msg)